In [ ]:
!pip install -r ../requirements.txt

In [ ]:
import os

DATA_ROOT = os.environ.get("DATA_ROOT", "../wildlife_dataset")

TRAIN_DIR = os.path.join(DATA_ROOT, "datawildlife_train")
VAL_DIR   = os.path.join(DATA_ROOT, "datawildlife_val")
TEST_DIR  = os.path.join(DATA_ROOT, "datawildlife_test")

In [ ]:
# create wildlife_detect.yaml from wildlife.yaml
BASE_YAML = os.environ.get("BASE_YAML", "../data/wildlife.yaml")
DETECT_YAML = "../data/wildlife_detect.yaml"

def to_yaml_path(p):
    return p.replace("\\", "/")

if os.path.exists(DETECT_YAML):
    print(f"Already exists, skip: {DETECT_YAML}")
else:
    with open(BASE_YAML, 'r') as f:
        base_content = f.read()

    detect_yaml = (
        f"train: {to_yaml_path(TRAIN_DIR)}\n"
        f"val: {to_yaml_path(VAL_DIR)}\n\n"
        f"{base_content}"
    )

    with open(DETECT_YAML, 'w') as f:
        f.write(detect_yaml)

    print(f"Created: {DETECT_YAML}")

In [ ]:
with open('../tools/train.py', 'r') as f:
    content = f.read()

content = content.replace(
    'torch.load(weights).get',
    'torch.load(weights, weights_only=False).get'
).replace(
    'torch.load(weights, map_location=device)',
    'torch.load(weights, map_location=device, weights_only=False)'
)

with open('../tools/train.py', 'w') as f:
    f.write(content)

In [ ]:
with open('../utils/loss.py', 'r') as f:
    content = f.read()

content = content.replace(
    'fg_mask_inboxes = matching_matrix.sum(0) > 0.0',
    'fg_mask_inboxes = (matching_matrix.sum(0) > 0.0).to(from_which_layer.device)'
)

with open('../utils/loss.py', 'w') as f:
    f.write(content)

In [ ]:
with open('../utils/metrics.py', 'r') as f:
    content = f.read()

content = content.replace('np.trapz', 'np.trapezoid')

with open('../utils/metrics.py', 'w') as f:
    f.write(content)

In [ ]:
with open('../data/hyp.scratch.tiny.yaml', 'r') as f:
    content = f.read()

content = content.replace('lr0: 0.01', 'lr0: 0.005')
content = content.replace('warmup_epochs: 3.0', 'warmup_epochs: 2.0')

with open('../data/hyp.scratch.tiny.yaml', 'w') as f:
    f.write(content)

In [ ]:
with open('../tools/tools/train.py', 'r') as f:
    content = f.read()
    
content = content.replace(
    "epochs += ckpt['epoch']  # finetune additional epochs",
    "epochs = opt.epochs  # use specified epochs for fine-tuning"
)

with open('../tools/tools/train.py', 'w') as f:
    f.write(content)

In [ ]:
#resume
%cd ..
import os
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'

!python tools/train.py --resume for-resume/stop_here/weights/last.pt